# Лекція 6: Основи веб-програмування на Python

**Курс**: Прикладна розробка програмного забезпечення на Python 2026
---

## Передумови

Ви вже знаєте:
- **Лекція 1**: Типи даних, змінні, f-рядки, venv, pip
- **Лекція 2**: Модель пам'яті, мутабельність, колекції, цикли та умови
- **Лекція 3**: Глибоке занурення в колекції, comprehensions, функції, args/kwargs
- **Лекція 4**: Lambda, генератори, декоратори, модулі, винятки, type hints
- **Лекція 5**: ООП, `@dataclass`, `@property`, JSON/CSV серіалізація, `pathlib`

> 💡 Класи та навички серіалізації в JSON з Лекції 5 — це концептуальна основа для **Pydantic-моделей** (Pydantic models) та **API-схем** (API schemas), які ми вивчимо сьогодні!

## Цілі навчання

Після цієї лекції ви зможете:

1. **Пояснити, що таке веб-сервер** (web server) і як працює цикл запит-відповідь (request-response)
2. **Використовувати правильні HTTP-методи** (methods) та коди стану (status codes) для CRUD-операцій
3. **Побудувати FastAPI-додаток** з роутерами (routers), Pydantic-схемами та автоматичною Swagger-документацією
4. **Ініціалізувати Python-проєкт** з `uv` та застосувати `ruff`/`black` для якості коду
5. **Пояснити, що таке MCP** (Model Context Protocol), назвати три його примітиви та сформулювати, навіщо він існує

## Вступ

У Лекції 5 ви побудували `ContactBook` — міні-проєкт, який зберігає дані в JSON-файл.
Але як **поділитися** цими даними з іншими програмами? Як зробити так, щоб мобільний додаток, веб-сторінка та чат-бот могли працювати з тими самими нотатками?

Відповідь: **веб-API** (Web API) — програмний інтерфейс, доступний через інтернет.

Сьогодні ми пройдемо повний шлях від "що таке сервер" до "запущений FastAPI-проєкт з документацією".

![API meme](https://http.cat/images/200.jpg)

---

## 1. Основи веб-серверів (Web Server Basics)

У цьому розділі ви зрозумієте, як працює зв'язок між клієнтом і сервером у вебі.

### 1.1 Клієнт-серверна архітектура (Client-Server Architecture)

Веб побудований на простій ідеї:

- **Клієнт** (client) — програма, яка *запитує* дані (браузер, мобільний додаток, curl, Python-скрипт)
- **Сервер** (server) — програма, яка *обробляє* запити та *повертає* відповіді

![Client-Server Architecture](https://developer.mozilla.org/en-US/docs/Learn_web_development/Getting_started/Web_standards/How_the_web_works/simple-client-server.png)

*Джерело: [MDN — How the Web Works](https://developer.mozilla.org/en-US/docs/Learn_web_development/Getting_started/Web_standards/How_the_web_works)*

> 🍽️ **Аналогія**: Ресторан!
> - Клієнт = відвідувач (замовляє страву)
> - Сервер = кухня (готує страву)
> - HTTP = офіціант (передає замовлення і приносить результат)
> - Меню = API-документація (що можна замовити)

### 1.2 Цикл запит-відповідь (Request-Response Cycle)

Кожна взаємодія з веб-сервером складається з **двох кроків**:

```
┌──────────┐         HTTP Request          ┌──────────┐
│          │  ──────────────────────────►   │          │
│  Client  │                               │  Server  │
│          │   ◄──────────────────────────  │          │
└──────────┘         HTTP Response          └──────────┘
```

1. Клієнт **надсилає запит** (request) — "дай мені список нотаток"
2. Сервер **обробляє** запит та **повертає відповідь** (response) — JSON з нотатками

![Request-Response Flow](https://mdn.github.io/shared-assets/images/diagrams/http/overview/fetching-a-page.svg)

*Джерело: [MDN — Overview of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview)*

### 1.3 Порти (Ports) та localhost

Коли ви запускаєте сервер, він **слухає** (listens) на певному **порту** (port).

> 🏢 **Аналогія**: IP-адреса = адреса будинку, порт = номер квартири.
> Щоб листоноша (клієнт) доставив пакунок, йому потрібен і будинок, і квартира.

#### Що таке `localhost` та `127.0.0.1`?

Кожен комп'ютер має **мережевий інтерфейс** (network interface) — «двері» для мережевого трафіку.
Серед них є спеціальний **loopback-інтерфейс** — він нікуди не виходить, а повертає трафік назад на ту саму машину.

| Термін | Значення | Коли використовувати |
|--------|----------|---------------------|
| `127.0.0.1` | Loopback IPv4-адреса | Завжди вказує на **цей** комп'ютер |
| `::1` | Loopback IPv6-адреса | Те саме, але для IPv6 |
| `localhost` | DNS-ім'я, яке резолвиться в `127.0.0.1` (або `::1`) | Зручний псевдонім |
| `0.0.0.0` | «Всі інтерфейси» — слухає на **всіх** мережевих адресах | Для доступу ззовні (Docker, LAN) |

```
┌─────────────────────────────────────────────────┐
│              Ваш комп'ютер                       │
│                                                  │
│   ┌──────────────┐    ┌─────────────────────┐   │
│   │ Браузер      │    │ FastAPI-сервер       │   │
│   │ (клієнт)     │    │ listening :8000      │   │
│   └──────┬───────┘    └──────────▲──────────┘   │
│          │  HTTP Request         │               │
│          └───────────────────────┘               │
│              loopback (127.0.0.1)                │
└─────────────────────────────────────────────────┘
```

> ⚠️ **`0.0.0.0` vs `127.0.0.1`** — важлива різниця!
> - `uvicorn main:app --host 127.0.0.1` → доступний **лише** з вашого комп'ютера
> - `uvicorn main:app --host 0.0.0.0` → доступний **з будь-якого** комп'ютера в мережі
> - У Docker-контейнерах **завжди** використовують `0.0.0.0`, бо `127.0.0.1` — це loopback *контейнера*, а не хост-машини!

#### Що таке порт?

**Порт** — це 16-бітне число (0–65535), яке ідентифікує конкретну програму на комп'ютері.
Це рівень **транспортного протоколу** (TCP/UDP) в моделі мережевих рівнів:

```
┌────────────────────────────┐
│  Application Layer         │  ← HTTP, WebSocket, gRPC
├────────────────────────────┤
│  Transport Layer (TCP/UDP) │  ← Порти живуть тут!
├────────────────────────────┤
│  Network Layer (IP)        │  ← IP-адреси (127.0.0.1)
├────────────────────────────┤
│  Link Layer (Ethernet/Wi-Fi)│  ← MAC-адреси, фізичне з'єднання
└────────────────────────────┘
```

*Це спрощена модель TCP/IP. У курсах з мереж ви вивчали її детальніше (модель OSI має 7 рівнів).*

| Частина URL | Значення | Приклад |
|-------------|----------|---------|
| `http://`   | Протокол (protocol) | HTTP або HTTPS |
| `localhost`  | IP-адреса (host) — "ця машина" | `127.0.0.1` |
| `:8000`     | Порт (port) | Де саме слухає сервер |
| `/notes`    | Шлях (path) | Який ресурс запитуємо |

```
http://localhost:8000/notes/create
└─┬──┘ └───┬───┘└─┬─┘└────┬─────┘
 протокол   хост  порт    шлях
```

#### Поширені порти

| Порт | Протокол / Сервіс | Примітка |
|------|-------------------|----------|
| `80` | HTTP | За замовчуванням (можна не вказувати в URL) |
| `443` | HTTPS | За замовчуванням для захищеного з'єднання |
| `8000` | FastAPI / uvicorn | Типовий для Python-розробки |
| `8080` | Альтернативний HTTP | Часто для проксі-серверів |
| `3000` | Node.js / React dev | Типовий для JS-розробки |
| `5432` | PostgreSQL | База даних |
| `6379` | Redis | Кеш / черга повідомлень |
| `27017` | MongoDB | NoSQL-база |

> 💡 **Чому порти 80 і 443 "невидимі" в URL?**
> Браузер автоматично додає `:80` для `http://` та `:443` для `https://`.
> Тому `https://google.com` — це насправді `https://google.com:443`.

> 🔗 **Зв'язок з іншими курсами**: Порти, IP-адреси та TCP/IP — це фундамент **мережевого програмування** (networking).
> Наприклад `socket` — низькорівневий Python-модуль, на якому побудовані HTTP-сервери.
> FastAPI та uvicorn абстрагують цю складність — але під капотом все працює через TCP-сокети!

---

## 2. Основи HTTP (HTTP Essentials)

**HTTP** (HyperText Transfer Protocol) — це протокол **прикладного рівня** (application layer), за яким спілкуються клієнт і сервер.
Кожен запит має **метод** (method), **URL**, **заголовки** (headers) і опціонально **тіло** (body).

#### Як HTTP працює «під капотом»?

HTTP побудований **поверх TCP** (Transmission Control Protocol):

```
┌─────────────────────────────────────────────────────────────────────┐
│                     Що відбувається при запиті                       │
│                                                                      │
│  1. DNS-резолюція: localhost → 127.0.0.1                             │
│  2. TCP-з'єднання: клієнт → SYN → сервер                            │
│                    клієнт ← SYN-ACK ← сервер   (TCP handshake)      │
│                    клієнт → ACK → сервер                             │
│  3. HTTP-запит:    клієнт → "GET /notes HTTP/1.1\r\n..." → сервер   │
│  4. HTTP-відповідь: клієнт ← "HTTP/1.1 200 OK\r\n..." ← сервер    │
│  5. TCP-закриття:  FIN → ACK → FIN → ACK                            │
└─────────────────────────────────────────────────────────────────────┘
```

> 💡 HTTP — це **текстовий** протокол (у версіях 1.0 та 1.1). Запити та відповіді — це звичайний текст, що передається через TCP-з'єднання. Це робить його легким для читання та дебагу.

#### HTTP vs HTTPS

| | HTTP | HTTPS |
|--|------|-------|
| **Порт** | 80 | 443 |
| **Шифрування** | ❌ Ні — дані передаються відкритим текстом | ✅ TLS (Transport Layer Security) |
| **Сертифікат** | Не потрібен | Потрібен SSL/TLS-сертифікат |
| **Для розробки** | ✅ `http://localhost:8000` | Зазвичай не потрібен локально |
| **У продакшені** | ❌ Ніколи! | ✅ Обов'язково! |

```
HTTP:   Client ── plain text ──► Server    😱 Хакер може прочитати!
HTTPS:  Client ── encrypted ──► Server     🔒 Захищено TLS
```

> 🔗 **Зв'язок з веб-розробкою**: Сучасні браузери позначають HTTP-сайти як «Not Secure».
> Для продакшен-серверів FastAPI зазвичай ставлять за **reverse proxy** (nginx, Caddy), який бере на себе TLS.

#### Еволюція HTTP

| Версія | Рік | Ключові зміни |
|--------|-----|---------------|
| HTTP/1.0 | 1996 | Одне з'єднання = один запит |
| **HTTP/1.1** | 1997 | Keep-alive з'єднання, chunked transfer ← **FastAPI використовує це** |
| **HTTP/2** | 2015 | Бінарний протокол, мультиплексування, server push |
| HTTP/3 | 2022 | Побудований на QUIC (UDP замість TCP), швидші з'єднання |


> 💡 **Для студентів**: Вам поки що достатньо знати HTTP/1.1 — це те, що ви побачите в `curl -v`.
> HTTP/2 і HTTP/3 — це оптимізації продуктивності, які працюють «прозоро» через reverse proxy.

### 2.1 HTTP-методи (HTTP Methods)

Метод визначає, **що** клієнт хоче зробити з ресурсом:

| Метод | CRUD-операція | Тіло запиту | Ідемпотентний | Безпечний |
|-------|---------------|:-----------:|:-------------:|:---------:|
| `GET` | Read (читати) | Ні | ✅ Так | ✅ Так |
| `POST` | Create (створити) | Так | ❌ Ні | ❌ Ні |
| `PUT` | Update — (повністю замінити) | Так | ✅ Так | ❌ Ні |
| `PATCH` | Update — (частково замінити) | Так | ❌ Ні | ❌ Ні |
| `DELETE` | Delete (видалити) | Ні | ✅ Так | ❌ Ні |

- **Ідемпотентний** (idempotent) = повторний виклик дає той самий результат. `PUT /notes/1` з тими самими даними — результат один.
- **Безпечний** (safe) = не змінює стан сервера. `GET` лише читає.

### 2.2 Коди стану (Status Codes)

Кожна відповідь сервера має числовий **код стану** — три цифри, що повідомляють, що відбулося:

| Діапазон | Категорія | Ключові коди для API |
|----------|-----------|---------------------|
| **2xx** | ✅ Успіх (Success) | `200 OK`, `201 Created`, `204 No Content` |
| **4xx** | ❌ Помилка клієнта (Client Error) | `400 Bad Request`, `404 Not Found`, `422 Unprocessable Entity` |
| **5xx** | 💥 Помилка сервера (Server Error) | `500 Internal Server Error` |

![HTTP 404](https://http.cat/images/404.jpg)

### 2.3 Анатомія HTTP-запиту та відповіді

**HTTP-запит** (Request):

```
POST /notes/create HTTP/1.1        ← Метод + Шлях + Версія
Host: localhost:8000                ← Заголовки (Headers)
Content-Type: application/json      ← Тип контенту
                                    ← Порожній рядок
{"title": "Моя нотатка"}            ← Тіло (Body)
```

![HTTP Request Format](https://mdn.github.io/shared-assets/images/diagrams/http/overview/http-request.svg)

**HTTP-відповідь** (Response):

```
HTTP/1.1 201 Created                ← Версія + Статус-код
Content-Type: application/json      ← Заголовки (Headers)
                                    ← Порожній рядок
{"id": "abc-123", "title": "Моя нотатка"}  ← Тіло (Body)
```

![HTTP Response Format](https://mdn.github.io/shared-assets/images/diagrams/http/overview/http-response.svg)

*Джерело: [MDN — HTTP Messages](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Messages)*

### 2.4 JSON як мова API

Ви вже знаєте JSON з Лекції 5 (`json.dumps`, `json.loads`).
У веб-API JSON — це **стандартний формат** обміну даними:

```json
{
  "title": "Моя нотатка",
  "content": "Привіт з FastAPI!",
  "tags": ["demo", "lecture6"]
}
```

Заголовок `Content-Type: application/json` повідомляє серверу: "тіло запиту — це JSON".

### 2.5 Path-параметри vs Query-параметри

Є два способи передати дані в URL:

| Тип | Синтаксис | Коли використовувати |
|-----|-----------|---------------------|
| **Path** (шлях) | `/notes/123` | Ідентифікація конкретного ресурсу |
| **Query** (запит) | `/notes?tag=demo&limit=10` | Фільтрація, пагінація, опціональні параметри |

```
/notes/123?format=json
└──┬──┘└┬┘ └────┬─────┘
  шлях  path   query
        param   param
```

### 2.6 Швидке знайомство з curl

`curl` — це командний інструмент для надсилання HTTP-запитів з терміналу.
Ви будете використовувати його для тестування API:

In [1]:
# curl — командний інструмент для HTTP-запитів
# Ці команди виконуються в ТЕРМІНАЛІ, не в Python!

# GET-запит (найпростіший):
# curl http://localhost:8000/health

# GET з деталями (-v = verbose):
# curl -v http://localhost:8000/health

# POST-запит з JSON-тілом:
# curl -X POST http://localhost:8000/notes/create \
#   -H "Content-Type: application/json" \
#   -d '{"title": "Test", "content": "Hello!"}'

# Давайте подивимось, як це працює на практиці (демо з httpbin):
import subprocess

result = subprocess.run(
    ["curl", "-s", "https://httpbin.org/get"],
    capture_output=True,
    text=True,
    timeout=10,
)
print(result.stdout[:500])  # Перші 500 символів відповіді

{
  "args": {}, 
  "headers": {
    "Accept": "*/*", 
    "Host": "httpbin.org", 
    "User-Agent": "curl/8.7.1", 
    "X-Amzn-Trace-Id": "Root=1-69aaaff8-4ef6b9ee4c3f02a2263c8363"
  }, 
  "origin": "88.154.17.79", 
  "url": "https://httpbin.org/get"
}



### 2.7 Не лише HTTP: інші протоколи у Python-екосистемі

HTTP — це фундамент веб-API, але він не єдиний протокол, який використовують Python-розробники.
Ось карта ключових протоколів та їх ролей:

#### Порівняння протоколів

| Протокол | Тип зв'язку | Формат даних | Коли використовувати | Python-бібліотеки |
|----------|-------------|-------------|---------------------|-------------------|
| **HTTP/REST** | Запит → Відповідь | JSON/XML | CRUD API, мікросервіси | FastAPI, Flask, Django REST |
| **WebSocket** | Двосторонній (bidirectional), постійне з'єднання | Текст/бінарний | Чати, live-нотифікації, real-time дашборди | `websockets`, FastAPI WebSocket |
| **gRPC** | Запит → Відповідь / стрімінг | Protocol Buffers (бінарний) | Міжсервісна комунікація, високий throughput | `grpcio`, `grpclib` |
| **GraphQL** | Запит → Відповідь | JSON | Гнучкі запити, мобільні клієнти | `strawberry`, `ariadne`, `graphene` |
| **MQTT** | Publish/Subscribe | Бінарний | IoT-пристрої, сенсори | `paho-mqtt` |
| **AMQP** | Черги повідомлень | Бінарний | Асинхронна обробка, мікросервіси | `aio-pika` (RabbitMQ) |

#### WebSocket — real-time комунікація

HTTP: «клієнт запитує → сервер відповідає → з'єднання закрите».
WebSocket: «з'єднання відкрите → обидва можуть надсилати повідомлення в будь-який час».

```
HTTP (polling):                          WebSocket:
┌────────┐    GET /updates    ┌────────┐  ┌────────┐  ← open →  ┌────────┐
│ Client │ ────────────────►  │ Server │  │ Client │ ◄═══════►  │ Server │
│        │ ◄──── "nothing" ── │        │  │        │  msg1 ───► │        │
│        │ ────────────────►  │        │  │        │ ◄─── msg2  │        │
│        │ ◄──── "nothing" ── │        │  │        │  msg3 ───► │        │
│        │ ────────────────►  │        │  │        │ ◄─── msg4  │        │
│        │ ◄──── "new data!"  │        │  │        │ ◄─── msg5  │        │
└────────┘  6 HTTP-запитів!   └────────┘  └────────┘ 1 з'єднання └────────┘
```

> 💡 **FastAPI підтримує WebSocket!** Це дозволяє будувати real-time функціональність (чати, live-оновлення)
> прямо в тому ж фреймворку, де ви робите REST API.

```python
# Приклад WebSocket в FastAPI (для ознайомлення):
from fastapi import FastAPI, WebSocket

app = FastAPI()

@app.websocket("/ws/chat")
async def chat(websocket: WebSocket):
    await websocket.accept()
    while True:
        data = await websocket.receive_text()   # Отримуємо від клієнта
        await websocket.send_text(f"Echo: {data}")  # Відправляємо назад
```

#### gRPC — швидкість між сервісами

Коли REST API не вистачає (мільйони запитів між мікросервісами), використовують **gRPC**:
- **Бінарний** формат (Protocol Buffers) замість текстового JSON — менший розмір, швидший парсинг
- **Стрімінг** — сервер може відправляти потік відповідей
- **Строга типізація** — .proto-файли описують API, як контракт

```
REST (JSON):   {"user_id": 123, "name": "Alice"}   ← ~40 байтів, текст
gRPC (Protobuf): \x08\x7b\x12\x05Alice             ← ~9 байтів, бінарний
```

> 🔗 **Де це зустрічається**: Google, Netflix, Spotify використовують gRPC для внутрішньої комунікації між сервісами.
> У Python-екосистемі gRPC часто зустрічається в ML-інфраструктурі (TensorFlow Serving, Triton).

#### Що обрати?

```
Будую публічний API для мобільного/веб-додатку?  → REST (FastAPI) ✅
Потрібен real-time (чат, нотифікації)?            → WebSocket
Мікросервіси спілкуються один з одним?            → gRPC
Клієнту потрібні гнучкі запити?                   → GraphQL
IoT / сенсори з обмеженим трафіком?               → MQTT
```

> 💡 **На цьому курсі** ми фокусуємось на **HTTP/REST** (FastAPI) — це найпоширеніший та найуніверсальніший підхід.
> Розуміння REST дає фундамент для вивчення решти протоколів.

---

## 3. Raw HTTP демо: `http.server`

Перед тим як перейти до FastAPI, давайте подивимося, що робить **найпростіший** веб-сервер.
Python має вбудований модуль `http.server` — він покаже, що сервер — це просто програма, яка слухає порт і відповідає на запити.

> ⚠️ **Увага**: Цей сервер — лише для демонстрації! Він НЕ підходить для реальних проєктів.

In [ ]:
# Мінімальний HTTP-сервер на стандартній бібліотеці Python
# Цей код показує СУТЬ того, що робить будь-який веб-сервер

from http.server import HTTPServer, BaseHTTPRequestHandler
import json


class SimpleHandler(BaseHTTPRequestHandler):
    """Обробник (handler) — вирішує, що відповісти на запит."""

    def do_GET(self):
        """Обробка GET-запитів."""
        # 1. Встановлюємо код відповіді
        self.send_response(200)
        # 2. Встановлюємо заголовки
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        # 3. Відправляємо тіло відповіді
        response = {"message": "Привіт з Python http.server!", "path": self.path}
        self.wfile.write(json.dumps(response, ensure_ascii=False).encode("utf-8"))


# Щоб запустити:
# server = HTTPServer(('localhost', 9000), SimpleHandler)
# print('Сервер слухає на http://localhost:9000')
# server.serve_forever()  # Блокує! Ctrl+C для зупинки

# Не запускаємо в Jupyter — це заблокує ядро!
# Запустіть в ОКРЕМОМУ терміналі, потім відкрийте http://localhost:9000 у браузері
print("Код сервера готовий — скопіюйте в .py файл і запустіть в терміналі!")
print("Або в терміналі: python -m http.server 9000")

### Що ми щойно побачили?

```
1. Сервер слухає порт 9000             → HTTPServer(('localhost', 9000), ...)
2. Клієнт (браузер) надсилає GET /     → do_GET(self)
3. Сервер обробляє та відповідає JSON  → send_response(200) + wfile.write(...)
```

Це **ВСЕ**, що робить веб-сервер. Решта — це зручності:
- Маршрутизація (routing): `/notes` → одна функція, `/health` → інша
- Валідація (validation): перевірка вхідних даних
- Серіалізація (serialization): автоматичне перетворення Python-об'єктів у JSON
- Документація (documentation): опис API

Уявіть, що вам треба все це написати вручну... 😰

> 💡 Саме тому існують **фреймворки** (frameworks) як FastAPI — вони автоматизують рутинну роботу.

---

## 4. Основи REST

**REST** (Representational State Transfer) — це **архітектурний стиль** (architectural style) для побудови веб-API.
Не стандарт, а набір принципів.

### 4.1 Ресурси як іменники (Resources as Nouns)

У REST ми працюємо з **ресурсами** (resources) — сутностями, які мають URL:

| ✅ Правильно (іменник) | ❌ Неправильно (дієслово) |
|-----------------------|--------------------------|
| `GET /notes` | `GET /getNotes` |
| `POST /notes` | `POST /createNote` |
| `DELETE /notes/123` | `POST /deleteNote?id=123` |

> 💡 URL описує **що** (ресурс), HTTP-метод описує **дію** (CRUD).

### 4.2 CRUD → HTTP маппінг

| CRUD | HTTP-метод | URL | Опис |
|------|-----------|-----|------|
| **C**reate | `POST` | `/notes` | Створити нову нотатку |
| **R**ead (список) | `GET` | `/notes` | Отримати всі нотатки |
| **R**ead (одна) | `GET` | `/notes/{id}` | Отримати нотатку за ID |
| **U**pdate | `PUT` | `/notes/{id}` | Оновити нотатку повністю |
| **D**elete | `DELETE` | `/notes/{id}` | Видалити нотатку |

### 4.3 Ідемпотентність (Idempotency)

**Ідемпотентна** (idempotent) операція — це операція, яку можна виконати кілька разів з тим самим результатом.

Чому це важливо? Уявіть, що мережа "засмикалась" і клієнт не отримав відповідь:

```
Клієнт → POST /notes {"title": "Привіт"} → ??? (таймаут)
Клієнт → POST /notes {"title": "Привіт"} → 201 Created
```

Результат: **два** "Привіт" нотатки! POST **не** ідемпотентний.

```
Клієнт → PUT /notes/1 {"title": "Привіт"} → ??? (таймаут)
Клієнт → PUT /notes/1 {"title": "Привіт"} → 200 OK
```

Результат: **одна** нотатка з title="Привіт". PUT ідемпотентний — безпечно повторити.

| Метод | Ідемпотентний? | Безпечно повторити? |
|-------|:--------------:|:-------------------:|
| GET | ✅ | ✅ Завжди |
| PUT | ✅ | ✅ Завжди |
| DELETE | ✅ | ✅ Завжди |
| POST | ❌ | ⚠️ Може створити дублікат |

### 4.4 Консистентна структура помилок

Хороший API повертає помилки в **однаковому форматі**, незалежно від типу помилки:

✅ **Правильно** — однакова структура:
```json
{
  "detail": "Note with id 'abc' not found",
  "error_code": "NOT_FOUND"
}
```

```json
{
  "detail": "Field 'title' is required",
  "error_code": "VALIDATION_ERROR"
}
```

❌ **Неправильно** — різні формати:
```json
{"error": "not found"}
```
```html
<html><body>500 Internal Server Error</body></html>
```

> 💡 Ми будемо використовувати структуру `{detail, error_code}` у нашому FastAPI-проєкті.

---

## 5. Основи FastAPI

[FastAPI](https://fastapi.tiangolo.com/) — сучасний Python-фреймворк для побудови API.
Він побудований на:
- **Starlette** — для обробки HTTP-запитів
- **Pydantic** — для валідації даних
- **Python type hints** — для автоматичної документації

Встановлення:
```bash
pip install fastapi uvicorn
# або з uv:
uv add fastapi uvicorn
```

### 5.1 Мінімальний FastAPI-додаток

In [ ]:
# Мінімальний FastAPI-додаток
# Збережіть як main.py та запустіть: uvicorn main:app --reload

from fastapi import FastAPI

app = FastAPI()  # Створюємо екземпляр (instance) додатку


@app.get("/")  # Декоратор (decorator) — пов'язує URL з функцією
def root():
    return {"message": "Привіт, FastAPI!"}  # Автоматично серіалізується в JSON


# Перевірка (в Jupyter — не запускає сервер, але код валідний)
print(f"App title: {app.title}")
print(f"Routes: {[route.path for route in app.routes]}")

### 5.2 HTTP-методи в FastAPI

FastAPI має декоратори для кожного HTTP-методу:

In [ ]:
from fastapi import FastAPI

app = FastAPI()


# GET — читання
@app.get("/items")
def get_items():
    return [{"id": 1, "name": "Item 1"}, {"id": 2, "name": "Item 2"}]


# POST — створення
@app.post("/items")
def create_item():
    return {"id": 3, "name": "New item", "created": True}


# DELETE — видалення
@app.delete("/items/{item_id}")
def delete_item(item_id: int):
    return {"deleted": item_id}


# Перевірка маршрутів
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"{route.methods} {route.path}")

### 5.3 Параметри шляху (Path Parameters)

In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/notes/{note_id}")
def get_note(note_id: int):  # note_id автоматично перетворюється в int!
    return {"note_id": note_id, "title": f"Note #{note_id}"}


# Якщо передати не число:
# GET /notes/abc → 422 Unprocessable Entity
# FastAPI автоматично валідує типи!

# Демо: виклик ендпоінту через TestClient (без запуску сервера)
try:
    from fastapi.testclient import TestClient

    client = TestClient(app)

    # Правильний запит
    response = client.get("/notes/42")
    print(f"GET /notes/42 → {response.status_code}: {response.json()}")

    # Неправильний тип
    response = client.get("/notes/abc")
    print(f"GET /notes/abc → {response.status_code}: validation error!")
except ImportError:
    print("httpx не встановлений — встановіть: pip install httpx")
    print("TestClient потребує httpx для роботи")

### 5.4 Query-параметри (Query Parameters)

In [ ]:
from fastapi import FastAPI

app = FastAPI()


@app.get("/notes")
def list_notes(skip: int = 0, limit: int = 10, tag: str | None = None):
    """
    Query-параметри — все, що після ? в URL:
    GET /notes?skip=0&limit=10&tag=demo
    """
    result = {"skip": skip, "limit": limit}
    if tag:
        result["filter_tag"] = tag
    return result


# Параметри зі значенням за замовчуванням — опціональні
# Параметри без default — обов'язкові

try:
    from fastapi.testclient import TestClient

    client = TestClient(app)
    response = client.get("/notes?skip=5&limit=20&tag=demo")
    print(f"GET /notes?skip=5&limit=20&tag=demo → {response.json()}")

    response = client.get("/notes")  # Все за замовчуванням
    print(f"GET /notes → {response.json()}")
except ImportError:
    print("httpx не встановлений — встановіть: pip install httpx")

### 5.5 Body-параметри (Request Body)

Для передачі складних даних (створення, оновлення) використовують **тіло запиту** (request body).
FastAPI використовує **Pydantic-моделі** для опису та валідації тіла:

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()


# Pydantic-модель для тіла запиту
class ItemCreate(BaseModel):
    name: str
    price: float
    in_stock: bool = True  # Значення за замовчуванням


@app.post("/items", status_code=201)  # 201 Created
def create_item(item: ItemCreate):  # FastAPI автоматично парсить JSON → ItemCreate
    return {"id": 1, **item.model_dump()}  # model_dump() → dict


try:
    from fastapi.testclient import TestClient

    client = TestClient(app)

    # Правильний запит
    response = client.post("/items", json={"name": "Laptop", "price": 999.99})
    print(f"POST /items → {response.status_code}: {response.json()}")

    # Невалідний запит (відсутнє обов'язкове поле)
    response = client.post("/items", json={"name": "Phone"})
    print(f"POST /items (no price) → {response.status_code}: validation error!")
except ImportError:
    print("httpx не встановлений — встановіть: pip install httpx")

### 5.6 Роутери (APIRouter) — організація коду

Коли API росте, всі ендпоінти в одному файлі — хаос. **Роутер** (router) дозволяє групувати пов'язані маршрути:

In [ ]:
from fastapi import APIRouter, FastAPI

# Роутер для здоров'я сервісу
health_router = APIRouter(tags=["health"])  # tags — для Swagger UI


@health_router.get("/health")
def health_check():
    return {"status": "ok"}


# Роутер для нотаток
notes_router = APIRouter(tags=["notes"])


@notes_router.get("/notes")
def list_notes():
    return []


@notes_router.post("/notes", status_code=201)
def create_note():
    return {"id": "abc-123"}


# Головний додаток — підключаємо роутери
app = FastAPI(title="Notes API")
app.include_router(health_router)  # Додає всі маршрути з health_router
app.include_router(notes_router)  # Додає всі маршрути з notes_router

# Результат: app має /health, /notes (GET), /notes (POST)
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"{route.methods} {route.path} → tags: {route.tags}")

---

## 6. Pydantic-схеми (Pydantic Schemas)

Пам'ятаєте `@dataclass` з Лекції 5? **Pydantic** `BaseModel` — це `@dataclass` **на стероїдах** 💪:

| Функція | `@dataclass` | Pydantic `BaseModel` |
|---------|-------------|---------------------|
| Автоматичний `__init__` | ✅ | ✅ |
| Автоматичний `__repr__` | ✅ | ✅ |
| Валідація типів | ❌ Ні (лише анотації) | ✅ Так (при створенні!) |
| Min/max length, ge/le | ❌ | ✅ `Field(min_length=1)` |
| Серіалізація в JSON | ❌ (вручну) | ✅ `.model_dump_json()` |
| OpenAPI/Swagger | ❌ | ✅ Автоматично |

### 6.1 Request-модель vs Response-модель

Чому потрібні **дві різні** моделі для одного ресурсу?

- **Request** (запит): клієнт надсилає `title`, `content` — але **не** `id` та `created_at` (їх генерує сервер)
- **Response** (відповідь): сервер повертає все: `id`, `title`, `content`, `created_at`

In [ ]:
from datetime import datetime, UTC
from uuid import uuid4

from pydantic import BaseModel, Field


# Request-модель: що клієнт НАДСИЛАЄ
class NoteCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)  # Валідація!
    content: str = Field(min_length=1)
    tags: list[str] = []  # Опціонально, за замовчуванням — порожній список


# Response-модель: що сервер ПОВЕРТАЄ
class NoteResponse(BaseModel):
    id: str  # Генерується сервером
    title: str
    content: str
    tags: list[str]
    created_at: datetime  # Генерується сервером


# Використання
note_input = NoteCreate(title="Моя нотатка", content="Привіт з FastAPI!")
print(f"Request: {note_input}")
print(f"Dict: {note_input.model_dump()}")

note_output = NoteResponse(
    id=str(uuid4()),
    title=note_input.title,
    content=note_input.content,
    tags=note_input.tags,
    created_at=datetime.now(UTC),
)
print(f"\nResponse: {note_output}")
print(f"JSON: {note_output.model_dump_json(indent=2)}")

### 6.2 Валідація (Validation) — автоматична 422

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class NoteCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    content: str = Field(min_length=1)
    tags: list[str] = []


# ✅ Валідний
note = NoteCreate(title="OK", content="Hello")
print(f"✅ Валідний: {note}")

# ❌ Порожній title
try:
    NoteCreate(title="", content="Hello")
except ValidationError as e:
    print(f"\n❌ Порожній title:")
    print(e)

# ❌ Відсутній content
try:
    NoteCreate(title="OK")
except ValidationError as e:
    print(f"\n❌ Відсутній content:")
    print(e)

# ❌ Title занадто довгий
try:
    NoteCreate(title="A" * 201, content="Hello")
except ValidationError as e:
    print(f"\n❌ Title > 200 символів:")
    print(e)

# FastAPI автоматично ловить ValidationError → повертає 422!

### 6.3 HTTPException — власні помилки

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()

# Фейкове сховище
NOTES_DB = {
    "1": {"id": "1", "title": "First note"},
    "2": {"id": "2", "title": "Second note"},
}


@app.get("/notes/{note_id}")
def get_note(note_id: str):
    if note_id not in NOTES_DB:
        # HTTPException — FastAPI перетворить це у відповідь з кодом 404
        raise HTTPException(
            status_code=404,
            detail=f"Note with id '{note_id}' not found",
        )
    return NOTES_DB[note_id]


try:
    from fastapi.testclient import TestClient

    client = TestClient(app)

    # Існуюча нотатка
    response = client.get("/notes/1")
    print(f"GET /notes/1 → {response.status_code}: {response.json()}")

    # Неіснуюча нотатка
    response = client.get("/notes/999")
    print(f"GET /notes/999 → {response.status_code}: {response.json()}")
except ImportError:
    print("httpx не встановлений — встановіть: pip install httpx")

### Вправа 1: Pydantic-схема для книги (10 хв)

Створіть Pydantic-моделі `BookCreate` та `BookResponse` зі такими полями:

**BookCreate** (запит):
- `title`: str, обов'язкове, 1–300 символів
- `author`: str, обов'язкове, 1–100 символів
- `year`: int, обов'язкове, від 1000 до 2030
- `genre`: str, опціональне, за замовчуванням `"unknown"`

**BookResponse** (відповідь):
- Всі поля з `BookCreate` + `id` (str) + `added_at` (datetime)

Перевірте валідацію: пустий title, рік 999, рік 2031.

In [ ]:
# Вправа 1: Ваш код тут
from pydantic import BaseModel, Field


class BookCreate(BaseModel):
    ...  # TODO: визначте поля з валідацією


class BookResponse(BaseModel):
    ...  # TODO: визначте поля


# Тести:
# book = BookCreate(title="Кобзар", author="Тарас Шевченко", year=1840)
# print(book)

<details>
<summary>Рішення (клікніть щоб побачити)</summary>

```python
from datetime import datetime, UTC
from uuid import uuid4
from pydantic import BaseModel, Field, ValidationError


class BookCreate(BaseModel):
    title: str = Field(min_length=1, max_length=300)
    author: str = Field(min_length=1, max_length=100)
    year: int = Field(ge=1000, le=2030)
    genre: str = "unknown"


class BookResponse(BaseModel):
    id: str
    title: str
    author: str
    year: int
    genre: str
    added_at: datetime


# Перевірка валідації
book = BookCreate(title="Кобзар", author="Тарас Шевченко", year=1840)
print(f"✅ {book}")

try:
    BookCreate(title="", author="Test", year=2000)
except ValidationError as e:
    print(f"❌ Пустий title: {e.error_count()} помилок")

try:
    BookCreate(title="Test", author="Test", year=999)
except ValidationError as e:
    print(f"❌ Рік 999: {e.error_count()} помилок")

try:
    BookCreate(title="Test", author="Test", year=2031)
except ValidationError as e:
    print(f"❌ Рік 2031: {e.error_count()} помилок")

# Response
resp = BookResponse(
    id=str(uuid4()),
    title=book.title,
    author=book.author,
    year=book.year,
    genre=book.genre,
    added_at=datetime.now(UTC),
)
print(f"\n✅ Response: {resp.model_dump_json(indent=2)}")
```

</details>

---

## 7. OpenAPI/Swagger + uvicorn

FastAPI **автоматично** генерує документацію для вашого API!

### 7.1 Автоматична документація

Коли ви запустите FastAPI-додаток, у вас є дві URL-адреси документації:

| URL | Формат | Опис |
|-----|--------|------|
| `/docs` | Swagger UI | Інтерактивна документація — можна тестувати прямо в браузері! |
| `/redoc` | ReDoc | Чиста документація для читання |
| `/openapi.json` | JSON | Машинозчитувана специфікація OpenAPI |

FastAPI генерує документацію з:
- **Type hints** вашого Python-коду
- **Pydantic-моделей** — поля, типи, валідації
- **Docstrings** — описи ендпоінтів
- **`response_model`** — схема відповіді

> 💡 Документація **завжди актуальна** — вона генерується з вашого коду, а не пишеться окремо!

### 7.2 Запуск з uvicorn

**uvicorn** — це ASGI-сервер (Asynchronous Server Gateway Interface), який запускає ваш FastAPI-додаток:

```bash
# Базовий запуск
uvicorn app.main:app

# Для розробки — з авто-перезавантаженням (auto-reload)
uvicorn app.main:app --reload

# Зміна порту (за замовчуванням 8000)
uvicorn app.main:app --reload --port 3000

# З uv:
uv run uvicorn app.main:app --reload
```

| Параметр | Опис |
|----------|------|
| `app.main:app` | Шлях до об'єкта FastAPI: `файл:змінна` |
| `--reload` | Автоматичне перезавантаження при зміні коду |
| `--port 3000` | Порт для прослуховування |
| `--host 0.0.0.0` | Слухати на всіх інтерфейсах (для Docker) |

---

## 8. Бутстрап проєкту (Project Bootstrap)

Тепер зберемо все разом у реальному проєкті! Ми використаємо `uv` — сучасний менеджер пакетів для Python.

> 📁 **Де живе проєкт?**
>
> Проєкт `notes-api` знаходиться в **`project/notes-api/`** (від кореня репозиторію курсу) і є спільним для всіх лекцій. Код нижче — це демонстрація того, як проєкт побудований. Під час або після лекції **скопіюйте та перепишіть** файли у свій власний репозиторій.

### 8.1 Ініціалізація проєкту з `uv`

```bash
# Крок 1: Створити новий проєкт
uv init notes-api
cd notes-api

# Крок 2: Додати залежності
uv add fastapi "uvicorn[standard]"

# Крок 3: Додати dev-залежності (інструменти розробки)
uv add --dev ruff black

# Крок 4: Встановити все
uv sync
```

Після цього у вас буде `pyproject.toml` — конфігурація проєкту:

In [ ]:
# Приклад pyproject.toml для нашого проєкту
pyproject_content = """
[project]
name = "notes-api"
version = "0.1.0"
requires-python = ">= 3.13"
dependencies = [
    "fastapi",
    "uvicorn[standard]",
]

[dependency-groups]
dev = [
    "ruff",
    "black",
]
"""
print(pyproject_content)

### 8.2 Структура проєкту

```
notes-api/
├── pyproject.toml           # Конфігурація проєкту (залежності)
├── app/
│   ├── __init__.py          # Пакет app
│   ├── main.py              # FastAPI app + include_router
│   ├── routers/
│   │   ├── __init__.py
│   │   ├── health.py        # GET /health
│   │   └── notes.py         # POST /notes/create, POST /notes/search
│   ├── schemas/
│   │   ├── __init__.py
│   │   ├── common.py        # HealthStatus, ErrorResponse
│   │   └── notes.py         # NoteCreate, NoteResponse, ...
│   ├── services/            # Порожній — для Лекції 7
│       └── __init__.py
```

| Директорія | Відповідальність |
|------------|------------------|
| `routers/` | HTTP-маршрути (що приймаємо, що повертаємо) |
| `schemas/` | Pydantic-моделі (валідація даних) |
| `services/` | Бізнес-логіка (порожній зараз — заповнимо в Лекції 7) |
| `clients/` | Клієнти до зовнішніх сервісів (порожній — для MCP в Лекції 7) |

### 8.3 Файли проєкту крок за кроком

Розглянемо кожен файл нашого проєкту `notes-api/`.

**`app/schemas/common.py`** — спільні схеми:

In [ ]:
# app/schemas/common.py
from pydantic import BaseModel


class HealthStatus(BaseModel):
    status: str  # Завжди "ok"
    version: str  # Версія API, напр. "0.1.0"


class ErrorResponse(BaseModel):
    detail: str  # Людиночитаний опис помилки
    error_code: str  # Машинозчитуваний код: "NOT_FOUND", "VALIDATION_ERROR"


# Демо
health = HealthStatus(status="ok", version="0.1.0")
error = ErrorResponse(detail="Note not found", error_code="NOT_FOUND")
print(f"Health: {health.model_dump()}")
print(f"Error: {error.model_dump()}")

**`app/schemas/notes.py`** — схеми для нотаток:

In [ ]:
# app/schemas/notes.py
from datetime import datetime

from pydantic import BaseModel, Field


class NoteCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    content: str = Field(min_length=1)
    tags: list[str] = []


class NoteResponse(BaseModel):
    id: str
    title: str
    content: str
    tags: list[str]
    created_at: datetime


class NoteSearchQuery(BaseModel):
    query: str = ""  # Пошуковий рядок
    tags: list[str] = []  # Фільтр за тегами
    limit: int = Field(default=10, ge=1, le=100)  # Макс. результатів: 1-100


class NoteSearchResult(BaseModel):
    results: list[NoteResponse]  # Список знайдених нотаток
    total: int  # Загальна кількість


print("Schemas defined ✅")

**`app/routers/health.py`** — ендпоінт здоров'я:

In [ ]:
# app/routers/health.py
from fastapi import APIRouter

# Імпортуємо зі schemas
# from app.schemas.common import HealthStatus

# Для демо в Jupyter використовуємо клас напряму
from pydantic import BaseModel


class HealthStatus(BaseModel):
    status: str
    version: str


router = APIRouter(tags=["health"])


@router.get("/health", response_model=HealthStatus)
async def health_check() -> HealthStatus:
    return HealthStatus(status="ok", version="0.1.0")


# Тест
import asyncio

result = asyncio.get_event_loop().run_until_complete(health_check())
print(f"Health check: {result.model_dump()}")

**`app/routers/notes.py`** — ендпоінти нотаток:

In [ ]:
# app/routers/notes.py
from datetime import UTC, datetime
from uuid import uuid4

from fastapi import APIRouter, status
from pydantic import BaseModel, Field


# Схеми (в реальному проєкті — import з app.schemas.notes)
class NoteCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    content: str = Field(min_length=1)
    tags: list[str] = []


class NoteResponse(BaseModel):
    id: str
    title: str
    content: str
    tags: list[str]
    created_at: datetime


class NoteSearchQuery(BaseModel):
    query: str = ""
    tags: list[str] = []
    limit: int = Field(default=10, ge=1, le=100)


class NoteSearchResult(BaseModel):
    results: list[NoteResponse]
    total: int


router = APIRouter(tags=["notes"])


@router.post(
    "/notes/create",
    response_model=NoteResponse,
    status_code=status.HTTP_201_CREATED,  # 201 — ресурс створено
)
async def create_note(note: NoteCreate) -> NoteResponse:
    """Створити нову нотатку (stub — повертає хардкодовані дані)."""
    return NoteResponse(
        id=str(uuid4()),
        title=note.title,
        content=note.content,
        tags=note.tags,
        created_at=datetime.now(UTC),
    )


@router.post("/notes/search", response_model=NoteSearchResult)
async def search_notes(query: NoteSearchQuery) -> NoteSearchResult:
    """Пошук нотаток (stub — завжди повертає порожній список)."""
    return NoteSearchResult(results=[], total=0)


# Тест
import asyncio

note_data = NoteCreate(title="Test note", content="Hello!", tags=["demo"])
result = asyncio.get_event_loop().run_until_complete(create_note(note_data))
print(f"Created: {result.model_dump_json(indent=2)}")

**`app/main.py`** — точка входу:

In [ ]:
# app/main.py — збирає все разом
# В реальному проєкті:

main_py_content = '''
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse

from app.routers import health, notes
from app.schemas.common import ErrorResponse

app = FastAPI(title="Notes API", version="0.1.0")

# Підключаємо роутери
app.include_router(health.router)
app.include_router(notes.router)


# Єдиний обробник помилок — консистентний формат!
@app.exception_handler(HTTPException)
async def http_exception_handler(
    request: Request, exc: HTTPException
) -> JSONResponse:
    return JSONResponse(
        status_code=exc.status_code,
        content=ErrorResponse(
            detail=str(exc.detail),
            error_code=f"HTTP_{exc.status_code}",
        ).model_dump(),
    )
'''
print(main_py_content)

### 8.4 Інструменти якості: ruff + black

| Інструмент | Що робить | Команда |
|------------|-----------|--------|
| **ruff** | Лінтер (linter) — знаходить помилки та проблеми стилю | `ruff check .` |
| **black** | Форматер (formatter) — автоматично форматує код | `black .` |

```bash
# Перевірка без змін
uv run ruff check .
uv run black --check .

# Автоматичне виправлення
uv run ruff check . --fix
uv run black .
```

> 💡 Запускайте `ruff` + `black` **перед кожним комітом** — ми побачимо, як налаштувати це автоматично  в Лекції 7!

### 8.5 Запуск та тестування

```bash
# 1. Перейти в директорію проєкту
cd notes-api

# 2. Встановити залежності
uv sync

# 3. Запустити сервер
uv run uvicorn app.main:app --reload

# 4. В іншому терміналі — тестування:

# Health check
curl http://localhost:8000/health
# → {"status": "ok", "version": "0.1.0"}

# Створити нотатку
curl -X POST http://localhost:8000/notes/create \
  -H "Content-Type: application/json" \
  -d '{"title": "My note", "content": "Hello from FastAPI!", "tags": ["demo"]}'
# → {"id": "...", "title": "My note", ...}

# Пошук (stub — завжди порожній)
curl -X POST http://localhost:8000/notes/search \
  -H "Content-Type: application/json" \
  -d '{"query": "test"}'
# → {"results": [], "total": 0}

# Swagger UI — відкрийте в браузері:
# http://localhost:8000/docs
```

### Вправа 2: Додати GET /notes/{note_id} (15 хв)

Додайте новий ендпоінт до `app/routers/notes.py`:

**Вимоги**:
1. `GET /notes/{note_id}` — приймає `note_id: str` як path-параметр
2. Якщо `note_id == "1"` → повертає хардкодований `NoteResponse` з кодом `200`
3. Для будь-якого іншого `note_id` → `HTTPException` з кодом `404` та повідомленням `"Note not found"`
4. `response_model=NoteResponse`

In [ ]:
# Вправа 2: Ваш код тут
# Додайте цю функцію в app/routers/notes.py

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel
from datetime import datetime


class NoteResponse(BaseModel):
    id: str
    title: str
    content: str
    tags: list[str]
    created_at: datetime


router = APIRouter()


@router.get("/notes/{note_id}", response_model=NoteResponse)
async def get_note(note_id: str):
    ...  # TODO: реалізуйте логіку
    pass

<details>
<summary>Рішення (клікніть щоб побачити)</summary>

```python
from datetime import UTC, datetime

from fastapi import APIRouter, HTTPException
from pydantic import BaseModel


class NoteResponse(BaseModel):
    id: str
    title: str
    content: str
    tags: list[str]
    created_at: datetime


router = APIRouter()


@router.get("/notes/{note_id}", response_model=NoteResponse)
async def get_note(note_id: str) -> NoteResponse:
    if note_id != "1":
        raise HTTPException(status_code=404, detail="Note not found")

    return NoteResponse(
        id="1",
        title="Hardcoded Note",
        content="This is a stub note for demo purposes.",
        tags=["demo", "stub"],
        created_at=datetime.now(UTC),
    )


# Тест (після додавання до проєкту):
# curl http://localhost:8000/notes/1
# → 200 {"id": "1", "title": "Hardcoded Note", ...}
#
# curl http://localhost:8000/notes/999
# → 404 {"detail": "Note not found"}
```

</details>

---

## 9. MCP — Model Context Protocol (Концептуальний вступ)

Ми навчились будувати API. Але як **AI-моделі** (Claude, ChatGPT, Copilot) підключаються до зовнішніх сервісів?

Кожна інтеграція — це окремий код, окремі формати, окремі протоколи... Хаос!

**MCP** (Model Context Protocol) вирішує цю проблему.

### 9.1 USB-C для AI


**MCP — це "USB-C" для AI-додатків:**

```
БЕЗ MCP:                               З MCP:
┌───────┐                              ┌───────┐
│Claude │──custom──►Google Keep         │Claude │
│       │──custom──►GitHub              │       │──MCP──►будь-який сервер
│       │──custom──►Slack               │       │
└───────┘                              └───────┘
┌───────┐                              ┌───────┐
│ChatGPT│──custom──►Google Keep         │ChatGPT│
│       │──custom──►GitHub              │       │──MCP──►будь-який сервер
└───────┘                              └───────┘

N хостів × M сервісів = N×M інтеграцій   N хостів + M серверів = N+M
```

> 💡 MCP перетворює проблему **N×M** (кожен AI з кожним сервісом) в проблему **N+M** (стандартний протокол).

### 9.2 Архітектура MCP: три учасники

![MCP Architecture](https://substackcdn.com/image/fetch/$s_!5Qxi!,w_1200,h_675,c_fill,f_jpg,q_auto:good,fl_progressive:steep,g_auto/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fdc01797d-9996-4b83-a00f-6771b8071d97_900x500.png)

*Джерело: [diamantai](https://diamantai.substack.com/p/model-context-protocol-mcp-explained)*

| Учасник | Роль | Приклади |
|---------|------|----------|
| **Host** (хост) | AI-додаток, що координує клієнтів | Claude Desktop, VS Code, Claude Code |
| **Client** (клієнт) | Компонент, що підтримує з'єднання з одним сервером | Створюється хостом для кожного сервера |
| **Server** (сервер) | Надає контекст через інструменти/ресурси/промпти | keep-mcp, filesystem, DB MCP |

```
┌─────────────────────────────────────┐
│           Host (Claude Desktop)      │
│                                      │
│  ┌──────────┐    ┌──────────┐       │
│  │ Client 1 │    │ Client 2 │       │
│  └────┬─────┘    └────┬─────┘       │
└───────┼───────────────┼─────────────┘
        │               │
   MCP Protocol    MCP Protocol
        │               │
   ┌────▼─────┐    ┌───▼──────┐
   │ Server A │    │ Server B │
   │(keep-mcp)│    │(filesystem)│
   └──────────┘    └──────────┘
```

### 9.3 Три примітиви MCP

MCP-сервер може надавати три типи можливостей:

| Примітив | Хто контролює | Опис | Приклад |
|----------|---------------|------|---------|
| **Tools** (інструменти) | Модель (model) | Функції, які AI може викликати | `create_note()`, `search()` |
| **Resources** (ресурси) | Додаток (app) | Джерела даних для контексту | Вміст файлів, записи БД |
| **Prompts** (промпти) | Користувач (user) | Шаблони для структурованих запитів | "Summarize this document" |

Протокол побудований на **JSON-RPC 2.0** — стандарт виклику функцій через JSON.

> 💡 Зверніть увагу на різницю: **Tools** = AI вирішує коли викликати, **Resources** = додаток вирішує що показати, **Prompts** = користувач вибирає шаблон.

### 9.4 Приклад: keep-mcp (Google Keep MCP Server)

[keep-mcp](https://github.com/feuerdev/keep-mcp) — це MCP-сервер, який підключає AI до **Google Keep** (нотатки).

**Інструменти (Tools) keep-mcp** — зверніть увагу на CRUD-маппінг з нашої лекції:

| CRUD | keep-mcp Tool | Що робить |
|------|---------------|----------|
| **C**reate | `create_note`, `create_list` | Створити нотатку/список |
| **R**ead | `find`, `get_note` | Знайти / отримати нотатку |
| **U**pdate | `update_note`, `set_note_color`, `pin_note` | Оновити нотатку |
| **D**elete | `trash_note`, `delete_note` | Видалити нотатку |

**Безпека за замовчуванням** (Safety by Default):
- За замовчуванням keep-mcp працює **тільки** з нотатками, які мають мітку `keep-mcp`
- Щоб працювати з усіма нотатками: `UNSAFE_MODE=true`

> ⚠️ Це чудовий приклад **безпечних значень за замовчуванням** (safe defaults) — тема, яку ми розглянемо глибше в Лекції 7.

**Конфігурація Claude Desktop** (для ознайомлення — ми НЕ запускаємо це зараз):

```json
{
  "mcpServers": {
    "keep-mcp": {
      "command": "pipx",
      "args": ["run", "keep-mcp"]
    }
  }
}
```

Після цієї конфігурації Claude зможе створювати, шукати та оновлювати ваші нотатки в Google Keep!

### 9.5 Чому це важливо для нас?

У **Лекції 7** ми:
1. Запустимо [keep-mcp](https://github.com/feuerdev/keep-mcp) — **MCP-сервер** для Google Keep та підключимо його до LLM-клієнта
2. Розширимо наш `notes-api` проєкт: async/await, httpx, конфігурація, тести
3. Налаштуємо повний **quality workflow**: ruff + black + pytest через Makefile

---

## Підсумок

### Що ми вивчили сьогодні:

**1. Веб-сервер та HTTP** — Розділи 1–3:
- Клієнт-серверна архітектура (client-server): запит (request) → відповідь (response)
- Порти (ports): IP = будинок, порт = квартира; `localhost:8000`
- `localhost` = `127.0.0.1` (loopback), `0.0.0.0` = всі інтерфейси (для Docker)
- Модель TCP/IP: Link → Network (IP) → Transport (TCP, порти) → Application (HTTP)
- HTTP-методи: GET (читати), POST (створити), PUT (оновити), DELETE (видалити)
- HTTP vs HTTPS: TLS-шифрування, сертифікати, reverse proxy в продакшені
- Еволюція: HTTP/1.1 (текстовий) → HTTP/2 (бінарний) → HTTP/3 (QUIC/UDP)
- Статус-коди: 2xx (успіх), 4xx (помилка клієнта), 5xx (помилка сервера)
- `http.server` — мінімальний сервер на stdlib (для розуміння суті)
- Інші протоколи: WebSocket (real-time), gRPC (міжсервісний), GraphQL (гнучкі запити)

**2. REST** — Розділ 4:
- Ресурси як іменники: `/notes`, `/notes/{id}` (не `/getNotes`!)
- CRUD → HTTP маппінг
- Ідемпотентність (idempotency): PUT/DELETE — безпечно повторити, POST — ні
- Консистентна структура помилок: `{detail, error_code}`

**3. FastAPI** — Розділи 5–7:
- `app = FastAPI()`, декоратори `@app.get()` / `@app.post()`
- Параметри: path (`/notes/{id}`), query (`?limit=10`), body (JSON + Pydantic)
- `APIRouter` для організації маршрутів
- Pydantic `BaseModel` — `@dataclass` на стероїдах: валідація, серіалізація, OpenAPI
- `HTTPException` для кастомних помилок
- Swagger UI на `/docs` — автоматична документація

**4. Проєкт** — Розділ 8:
- `uv init` + `uv add` + `uv sync` — менеджмент проєкту
- Структура: `app/routers/`, `schemas/`, `services/`, `clients/`
- `ruff check .` + `black .` — якість коду

**5. MCP** — Розділ 9:
- MCP = "USB-C для AI" — стандартний протокол підключення AI до інструментів
- Три учасники: Host → Client → Server
- Три примітиви: Tools (модель), Resources (додаток), Prompts (користувач)
- keep-mcp: конкретний приклад MCP-сервера для Google Keep

---

## Що далі?

### Лекція 7: Python Web Server Integrations: Async, HTTPX, Testing, Practical MCP

Ваш FastAPI-проєкт — це скелет. У Лекції 7 ми оживимо його!

- **Async**: event loop, `async`/`await` — чому FastAPI їх використовує
- **httpx**: виклик зовнішніх API з Python (sync та async)
- **Конфігурація**: `pydantic-settings` + `.env` — типізовані налаштування
- **Практичний MCP**: налаштування keep-mcp для Google Keep + підключення до LLM
- **Тестування**: `pytest` + `TestClient` + мокінг через `monkeypatch`
- **Quality workflow**: `Makefile` з `make check` = ruff + black + pytest

> 💡 Підказка: ваші Pydantic-схеми `NoteCreate` та `NoteResponse` — готова основа. `services/` та `clients/` — директорії, які ми заповнимо!

### Домашнє завдання

**Завдання 1 — Новий ендпоінт**:
Додайте `DELETE /notes/{note_id}` до `app/routers/notes.py`:
- Приймає `note_id: str`
- Якщо `note_id == "1"` → повертає `204 No Content`
- Інакше → `404 Not Found` з `ErrorResponse`

**Завдання 2 — Нова схема**:
Створіть `NoteUpdate(BaseModel)` з опціональними полями `title`, `content`, `tags`
(використайте `str | None = None`).
Додайте `PUT /notes/{note_id}` що приймає `NoteUpdate` та повертає `NoteResponse`.

**Завдання 3 — ruff + black**:
Запустіть `ruff check .` та `black --check .` на вашому проєкті.
Виправте всі помилки та переконайтесь, що обидва проходять без помилок.

---

## Джерела

### HTTP та Веб
- [MDN — Overview of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview) — Повний огляд протоколу HTTP
- [MDN — Evolution of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Evolution_of_HTTP) — Від HTTP/1.0 до HTTP/3
- [MDN — HTTP Methods](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods) — Довідник HTTP-методів
- [MDN — HTTP Status Codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status) — Усі коди стану
- [MDN — How the Web Works](https://developer.mozilla.org/en-US/docs/Learn_web_development/Getting_started/Web_standards/How_the_web_works) — Як працює веб
- [MDN — An overview of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview) — HTTP як протокол прикладного рівня

### HTTPS та безпека
- [MDN — Transport Layer Security (TLS)](https://developer.mozilla.org/en-US/docs/Web/Security/TLS) — Як працює шифрування
- [How HTTPS Works](https://howhttps.works/) — Інтерактивний комікс про HTTPS

### Мережі та протоколи
- [MDN — WebSockets API](https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API) — Документація WebSocket
- [gRPC — Official Docs](https://grpc.io/docs/) — Документація gRPC
- [GraphQL — Official Docs](https://graphql.org/learn/) — Вступ до GraphQL

### FastAPI та Pydantic
- [FastAPI — Official Tutorial](https://fastapi.tiangolo.com/tutorial/) — Офіційний туторіал
- [FastAPI — WebSockets](https://fastapi.tiangolo.com/advanced/websockets/) — WebSocket в FastAPI
- [FastAPI — Bigger Applications](https://fastapi.tiangolo.com/tutorial/bigger-applications/) — Структура великих додатків
- [Pydantic — Official Docs](https://docs.pydantic.dev/latest/) — Документація Pydantic v2
- [Pydantic — Field Validators](https://docs.pydantic.dev/latest/concepts/validators/) — Валідація полів

### Інструменти
- [uv — Official Docs](https://docs.astral.sh/uv/) — Менеджер пакетів uv
- [ruff — Official Docs](https://docs.astral.sh/ruff/) — Лінтер ruff
- [Black — Official Docs](https://black.readthedocs.io/en/stable/) — Форматер Black

### MCP
- [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol) - Анонс MCP від Anthropic
- [MCP — Official Docs](https://modelcontextprotocol.io/) — Офіційна документація MCP
- [MCP — Architecture](https://modelcontextprotocol.io/docs/learn/architecture) — Архітектура протоколу
- [keep-mcp](https://github.com/feuerdev/keep-mcp) — MCP-сервер для Google Keep